# Packages

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm
from pygam import GAM, s

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
%load_ext nb_black

# Data

In [ ]:
# World record marathon dataset (from wikipedia: https://en.wikipedia.org/wiki/Marathon_world_record_progression)
# Thanks to chatgpt for the format!
records = [
    (1908, "2:55:18.4"),
    (1909, "2:52:45.4"),
    (1909, "2:46:52.8"),
    (1909, "2:46:04.6"),
    (1909, "2:42:31.0"),
    (1909, "2:40:34.2"),
    (1913, "2:38:16.2"),
    (1913, "2:36:06.6"),
    (1914, "2:38:00.8"),
    (1920, "2:32:35.8"),
    (1925, "2:29:01.8"),
    (1929, "2:30:57.6"),
    (1935, "2:26:14"),
    (1935, "2:27:49.0"),
    (1935, "2:26:44.0"),
    (1935, "2:26:42"),
    (1947, "2:25:39"),
    (1952, "2:20:42.2"),
    (1953, "2:18:40.4"),
    (1953, "2:18:34.8"),
    (1954, "2:17:39.4"),
    (1956, "2:18:04.8"),
    (1958, "2:15:17.0"),
    (1960, "2:15:16.2"),
    (1963, "2:15:15.8"),
    (1963, "2:14:28"),
    (1963, "2:14:43"),
    (1964, "2:13:55"),
    (1964, "2:12:11.2"),
    (1965, "2:12:00"),
    (1967, "2:09:36.4"),
    (1969, "2:08:33.6"),
    (1970, "2:09:28.8"),
    (1974, "2:09:12"),
    (1978, "2:09:05.6"),
    (1980, "2:09:01"),
    (1981, "2:08:18"),
    (1984, "2:08:05"),
    (1985, "2:07:12"),
    (1988, "2:06:50"),
    (1998, "2:06:05"),
    (1999, "2:05:42"),
    (2002, "2:05:38"),
    (2003, "2:04:55"),
    (2007, "2:04:26"),
    (2008, "2:03:59"),
    (2011, "2:03:38"),
    (2013, "2:03:23"),
    (2014, "2:02:57"),
    (2018, "2:01:39"),
    (2022, "2:01:09"),
    (2023, "2:00:35"),
]

In [ ]:
# Convert to dataframe
data = pd.DataFrame(records, columns=["Year", "Time"])

In [ ]:
# Convert time string to minutes
def time_to_minutes(t):
    parts = t.split(":")
    h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
    return h * 60 + m + s / 60

data["Minutes"] = data["Time"].apply(time_to_minutes)

In [ ]:
# Take the best time per year
data_best = data.groupby('Year')['Minutes'].min().reset_index()

In [ ]:
# Ensure we have every year in our dataframe
data_full = pd.DataFrame({'Year': range(data_best['Year'].min(), data_best['Year'].max()+1)})
data_full = data_full.merge(data_best, on='Year', how='left')
data_full['Minutes'] = data_full['Minutes'].ffill()

In [ ]:
fig = px.line(data_full, x="Year", y="Minutes",
              title="Men's Marathon World Record in Minutes (1908–2025)",
              labels={"YEAR": "Year", "Minutes": "World Record Time (minutes)"},
              template="simple_white")

fig.show()

In [ ]:
# Split into training and test sets
train = data_full[data_full['Year'] <= 2005]
test = data_full[data_full['Year'] > 2005]

# ARIMA Forecast

## Stationarity Check

In [ ]:
def adf_test(series):
    test_results = adfuller(series)
    print('ADF Statistic: ', test_results[0])
    print('P-Value: ', test_results[1])
    print('Critical Values:')
    for thres, adf_stat in test_results[4].items():
        print('\t%s: %.2f' % (thres, adf_stat))


In [ ]:
# Check the stationarity of the original data
adf_test(train['Minutes'])

In [ ]:
# Stabilise the variance of our data using a log transform
train.loc[:, 'Minutes_log'] = np.log(train['Minutes'])

## Model

In [ ]:
# Train a regular baseline ARIMA model
model_arima = pm.auto_arima(train['Minutes_log'],
                            seasonal=False,
                            stepwise=True,
                            suppress_warnings=True,
                            trace=True)

## Forecast

In [ ]:
# Forecast on our training data
n_periods = len(test)
forecast_arima = np.exp(model_arima.predict(n_periods=n_periods))

## Analysis

In [ ]:
mae = mean_absolute_error(test['Minutes'], forecast_arima)
rmse = np.sqrt(mean_squared_error(test['Minutes'], forecast_arima))

print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
# Plot the forecast along with the real data we observe
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test['Year'],
    y=forecast_arima,
    mode='lines',
    name='Forecast ARIMA'
))

fig.add_trace(go.Scatter(
    x=test['Year'],
    y=test['Minutes'],
    mode='lines',
    name='Test',
))

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='World Record Time (minutes)',
    legend_title='Legend',
    template="simple_white",
    width=900,
    height=500
)

fig.show()


# GAM and Splines

## Model

In [ ]:
# Prepare data
X_train = train['Year'].values
y_train = train['Minutes'].values

In [ ]:
# Fit a linear spline GAM
gam = GAM(s(0, spline_order=3, n_splines=20)).fit(X_train, y_train)

In [ ]:
# Predictions on the training years
X_fit = np.linspace(X_train.min(), X_train.max(), 200).reshape(-1,1)
y_fit = gam.predict(X_fit)

In [ ]:
# Get the knots
n_splines = gam.terms[0].n_splines
knots = np.linspace(X_train.min(), X_train.max(), n_splines)

In [ ]:
# Create interactive plot
fig = go.Figure()

# Training points
fig.add_trace(go.Scatter(
    x=X_train, y=y_train, mode='markers',
    marker=dict(color='blue', size=8),
    name='Train data'
))

# GAM fit line
fig.add_trace(go.Scatter(
    x=X_fit.flatten(), y=y_fit, mode='lines',
    line=dict(color='red', width=2),
    name='GAM Linear Spline Fit'
))

# Add knots as vertical dashed lines
for k in knots:
    fig.add_trace(go.Scatter(
        x=[k, k], y=[y_train.min()-1, y_train.max()+1],
        mode='lines',
        line=dict(color='black', width=1, dash='dash'),
        name=f'Knot {k}',
        showlegend=False  # hide legend for multiple knots
    ))

# Layout
fig.update_layout(
    title="GAM Spline Fit with Knots (Training Data)",
    xaxis_title="Year",
    yaxis_title="Marathon Time (Minutes)",
    template="plotly_white"
)

fig.show()

## Forecast

In [ ]:
# Predict on the test years
X_test = test['Year'].values
forecast_gam = gam.predict(X_test)

## Analysis

In [ ]:
mae = mean_absolute_error(test['Minutes'], forecast_gam)
rmse = np.sqrt(mean_squared_error(test['Minutes'], forecast_gam))

print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
# Plot the forecast along with the real data we observe
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test['Year'],
    y=forecast_arima,
    mode='lines',
    name='Forecast ARIMA'
))

fig.add_trace(go.Scatter(
    x=test['Year'],
    y=forecast_gam,
    mode='lines',
    name='Forecast GAM'
))

fig.add_trace(go.Scatter(
    x=test['Year'],
    y=test['Minutes'],
    mode='lines',
    name='Test',
))

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='World Record Time (minutes)',
    legend_title='Legend',
    template="simple_white",
    width=900,
    height=500
)

fig.show()
